# Detection paper: GPU experiments

Runs the three GPU-bound items the paper still needs. Each section is independent and
checkpoints to Drive, so they can be run across separate Colab sessions.

| § | Experiment | Answers | Rough runtime (L4/A100) |
|---|---|---|---|
| 2 | Matched-budget seed reruns, 3 seeds x 2 architectures | "the architecture comparison is confounded" | 4-6 h |
| 3 | Model-based positive control on the manipulated corpus | "the screen was validated on the case that motivated it" | 1-1.5 h |
| 4 | Bootstrap confidence intervals on the benchmark | "no error bars anywhere" | minutes (CPU) |

**Expected layout on Drive** (`MyDrive/evb/`):

```
evb/
  images/            <- the ~4 GB image corpus, train/ val/ test/ (filenames match labels)
  labels/            <- copy of data/labels_release/detector  (train/ val/ test/)
  labels_convshift/  <- copy of data/detector_convshift        (train/ val/ test/)
  runs/              <- created by this notebook
```

`labels/` and `labels_convshift/` are in the repo and are small. Only `images/` needs
uploading, and only once.

> **Before running Sections 3 and 4.** These evaluate on the multi-source `val` split,
> because the repository's `test/` directory holds only the 43-image single-source
> split, all of it from the convention-divergent source. Scoring a cross-facility claim
> on that would be meaningless.
>
> The recovered corpus currently covers **43 of 292 val images**. Restore the rest
> before running Section 3, using the Roboflow fallback for the two short sources only:
>
> ```python
> SOURCES = [("ev-battery", "ev-battery-component-detection-gqljq", 1),
>            ("academic-lsrwt", "ev-battery-components-edfw3", 1)]
> ```
>
> Section 2 is unaffected and can run now: it trains on `train`, which is complete, and
> epoch selection against `val` is disabled.


> **Why a fixed epoch budget and `last.pt`.** The validation split available in this
> session is small and drawn from the convention-divergent source, so epoch selection
> against it would bias every model toward the annotation convention this paper
> identifies as the outlier. Early stopping is therefore disabled
> (`patience = EPOCHS + 1`) and the final checkpoint is used rather than the
> val-selected one. This is also what the matched-budget comparison requires: both
> architectures must see exactly the same number of epochs for the margin between
> them to be attributable to architecture. Validation still runs each epoch for
> logging; it simply does not drive selection.


## 0. Get the data into this session

The image corpus is ~4 GB and is not in the repository. Rather than uploading it to
Drive, it is downloaded straight from Roboflow Universe into the Colab VM, which is a
datacenter-to-datacenter transfer and takes a few minutes. A free Roboflow API key is
required: sign in at https://app.roboflow.com and copy the key from Settings.

The labels come from the repository, so nothing needs uploading at all. The assembled
corpus is cached to Drive at the end of this section, so later sessions skip the
download.

In [ ]:
# Repository: labels, manipulated labels, provenance, tooling
!git clone -q https://github.com/Komil-jon/ev-battery-vision-pipeline.git /content/repo || git clone -q https://github.com/Komil-jon/research.git /content/repo
import pathlib
REPO = pathlib.Path('/content/repo')
assert REPO.exists(), 'clone failed - check the repository URL'
print('labels        :', len(list((REPO/'data/labels_release/detector').rglob('*.txt'))), 'files')
print('labels_shifted:', len(list((REPO/'data/detector_convshift').rglob('*.txt'))), 'files')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/evb')
(ROOT/'runs').mkdir(parents=True, exist_ok=True)
IMG_CACHE = ROOT/'images'
print('image cache present:', IMG_CACHE.exists(),
      f'({len(list(IMG_CACHE.rglob("*"))) if IMG_CACHE.exists() else 0} entries)')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Prefer a corpus already on Drive. ev_diverse_data.zip from an earlier run is very
# likely the 4,425-image multi-source corpus this notebook needs.
import zipfile, pathlib, shutil, collections

CANDIDATES = sorted(
    list(pathlib.Path('/content/drive/MyDrive').glob('*.zip')) +
    list(pathlib.Path('/content/drive/MyDrive').glob('*/*.zip')),
    key=lambda p: -p.stat().st_size)[:12]

LABELS_ORIG  = REPO/'data/labels_release/detector'
LABELS_SHIFT = REPO/'data/detector_convshift'
stems = set()
for sp in ('train','val','test'):
    stems |= {q.stem for q in (LABELS_ORIG/sp).glob('*.txt')}
print(f'{len(stems)} label stems to satisfy\n')

print(f'{"archive":<34}{"MB":>8}{"images":>9}{"matching stems":>16}')
report = {}
for z in CANDIDATES:
    try:
        with zipfile.ZipFile(z) as zf:
            names = [n for n in zf.namelist()
                     if n.lower().endswith(('.jpg','.jpeg','.png')) and not n.startswith('__MACOSX')]
            hit = sum(1 for n in names if pathlib.PurePath(n).stem in stems)
    except Exception as e:
        print(f'{z.name:<34}{"":>8}  unreadable: {type(e).__name__}'); continue
    report[z] = (len(names), hit)
    print(f'{z.name:<34}{z.stat().st_size/1e6:>8.0f}{len(names):>9}{hit:>16}')

BEST = max(report, key=lambda k: report[k][1]) if report else None
if BEST and report[BEST][1] > 0.6*len(stems):
    print(f'\nUsing {BEST.name}: {report[BEST][1]} of {len(stems)} stems '
          f'({100*report[BEST][1]/len(stems):.1f}%). No Roboflow download needed.')
else:
    print('\nNo archive covers the corpus. Run the Roboflow fallback cell below.')

In [ ]:
IMAGES = pathlib.Path('/content/images'); IMAGES.mkdir(exist_ok=True)

if IMG_CACHE.exists() and len({q.stem for q in IMG_CACHE.glob('*')} & stems) > 0.9*len(stems):
    print('Drive cache covers the corpus; copying from cache instead of unzipping.')
    !cp -n "{IMG_CACHE}"/* /content/images/ 2>/dev/null
else:
    for z in ([BEST] if BEST else []) + [k for k in report if k is not BEST]:
        with zipfile.ZipFile(z) as zf:
            for n in zf.namelist():
                if not n.lower().endswith(('.jpg','.jpeg','.png')) or n.startswith('__MACOSX'):
                    continue
                if pathlib.PurePath(n).stem not in stems:
                    continue
                tgt = IMAGES/pathlib.PurePath(n).name
                if tgt.exists():
                    continue
                with zf.open(n) as s_, open(tgt, 'wb') as o_:
                    shutil.copyfileobj(s_, o_)

have = {q.stem for q in IMAGES.glob('*')}
matched = len(have & stems)
print(f'\ntotal: {matched} / {len(stems)}  ({100*matched/len(stems):.1f}%)\n')

for sp in ('train','val','test'):
    need_sp = {q.stem for q in (LABELS_ORIG/sp).glob('*.txt')}
    got_sp  = need_sp & have
    flag = '' if len(got_sp) == len(need_sp) else '  <-- incomplete'
    print(f'  {sp:<6}{len(got_sp):>5} / {len(need_sp):<5}{flag}')

MTECH = ('aug_','tesla_model','chevrolet-bolt','nissan-leaf')
PRE = {'gqljq':'roboflow_ev-battery-component-detection-gqljq',
       'edfw3':'roboflow_ev-battery-components-edfw3','ybmvt':'ev-battery-sample-ybmvt',
       'final_mobilenet':'final_mobilenet_results','automated':'automated-disassembly',
       'ue_rav4':'ue_rav4_module','battery_comp':'battery_comp','last_exp4':'last_exp',
       'bmw_i3':'bmw_i'}
def src_of(n):
    n = n.lower()
    for s_, q_ in PRE.items():
        if n.startswith(q_): return s_
    return 'mtech' if any(n.startswith(q_) for q_ in MTECH) else 'other'

miss = collections.Counter(src_of(s_) for s_ in (stems - have))
print('\nmissing by source:', dict(miss) or 'none')

train_ok = len({q.stem for q in (LABELS_ORIG/'train').glob('*.txt')} & have)
test_ok  = len({q.stem for q in (LABELS_ORIG/'test').glob('*.txt')} & have)
if train_ok < 0.98*4425 or test_ok < 43:
    print('\nSTOP: train or test is incomplete. Results computed on a partial corpus '
          'are not comparable with the paper. Run the Roboflow fallback below.')
else:
    print('\nTrain and test are complete. Gaps in val do not affect these experiments: '
          'epoch selection is disabled in Section 2 for exactly that reason.')

#### Roboflow fallback

Only needed if the cell above reports a low match rate, and then only for the sources
it lists as short. A free key is taken from the form field at runtime and is never
written to disk or committed. If you have pasted a key anywhere public, regenerate it
first at app.roboflow.com.

In [ ]:
#@title Roboflow download  { display-mode: "form" }
ROBOFLOW_API_KEY = ""  #@param {type:"string"}

SOURCES = [
    ("ev-battery",         "ev-battery-component-detection-gqljq", 1),
    ("academic-lsrwt",     "ev-battery-components-edfw3",          1),
    ("ca-2kt9o",           "last_exp4",                            1),
    ("mtech-project-ohj8a","ev-battery-pack",                      1),
    ("ca-2kt9o",           "battery_comp",                         1),
    ("uerymnd",            "ue_d1_defect_detection",               1),
    ("tina-eslami",        "final_mobilenet_results",              1),
    ("auto-dissasembly",   "automated-disassembly",                1),
    ("uerymnd",            "bmw_i3",                               1),
    ("uerymnd",            "ue_rav4_module",                       1),
    ("ev-battery",         "ev-battery-sample-ybmvt",              1),
    ("tina-eslami",        "battery-modules",                      1),
    ("rabds",              "radbs-yda1j",                          2),
]

import os, shutil
RAW = pathlib.Path('/content/raw'); RAW.mkdir(exist_ok=True)

if IMG_CACHE.exists() and len(list(IMG_CACHE.rglob('*.jpg'))) > 3000:
    print('Drive cache already populated; skipping download.')
else:
    assert ROBOFLOW_API_KEY, 'paste a Roboflow API key into the form field above'
    !pip -q install roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for ws, proj, ver in SOURCES:
        dest = RAW/proj
        if dest.exists():
            print('have', proj); continue
        try:
            rf.workspace(ws).project(proj).version(ver).download('yolov8', location=str(dest))
            n = len(list(dest.rglob('*.jpg'))) + len(list(dest.rglob('*.png'))) + len(list(dest.rglob('*.jpeg')))
            print(f'{proj:<40} {n} images')
        except Exception as e:
            print(f'{proj:<40} FAILED: {type(e).__name__}: {e}')
    print('\\nNote: a source that fails here is usually one whose owner has since made it '
          'private. The corpus tolerates a missing minor source; report which ones failed.')

In [ ]:
IMG_CACHE = ROOT/'images'
if matched and (not IMG_CACHE.exists() or len(list(IMG_CACHE.glob('*'))) < matched):
    IMG_CACHE.mkdir(parents=True, exist_ok=True)
    !cp -n /content/images/* "{IMG_CACHE}/" 2>/dev/null
print('Drive cache:', len(list(IMG_CACHE.glob('*'))) if IMG_CACHE.exists() else 0, 'images')
for n, q in (('images', IMAGES), ('labels', LABELS_ORIG), ('labels_convshift', LABELS_SHIFT)):
    print(('OK   ' if q.exists() else 'MISS ') + f'{n:<18}{q}')

## 1. Environment

In [ ]:
!pip -q install ultralytics==8.3.0 supervision==0.25.1
!pip -q install "rfdetr[train]"
import ultralytics, supervision, torch
print('ultralytics', ultralytics.__version__, '| supervision', supervision.__version__,
      '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
!pip -q install ultralytics==8.3.0 supervision==0.25.1
!pip -q install "rfdetr[train]"
import ultralytics, supervision, torch
print('ultralytics', ultralytics.__version__, '| supervision', supervision.__version__,
      '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

### Assemble a YOLO-format dataset

Images live in one tree and labels in another, so they are linked into the layout
Ultralytics expects. `LABELS` selects the original corpus or the manipulated one used
for the positive control in §3.

In [ ]:
import shutil, yaml

def build_dataset(labels_dir, tag):
    """Link images + labels into /content/ds_<tag> in Ultralytics layout."""
    dst = pathlib.Path(f'/content/ds_{tag}')
    if dst.exists():
        shutil.rmtree(dst)
    n_img = n_lab = 0
    for split in ('train', 'val', 'test'):
        (dst / split / 'images').mkdir(parents=True, exist_ok=True)
        (dst / split / 'labels').mkdir(parents=True, exist_ok=True)
        src_lab = pathlib.Path(labels_dir) / split
        if not src_lab.exists():
            continue
        stems = {p.stem for p in src_lab.glob('*.txt')}
        for p in IMAGES.rglob('*'):
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png') and p.stem in stems:
                os.symlink(p, dst / split / 'images' / p.name)
                n_img += 1
        for p in src_lab.glob('*.txt'):
            os.symlink(p, dst / split / 'labels' / p.name)
            n_lab += 1
    cfg = dst / 'data.yaml'
    cfg.write_text(yaml.dump({
        'path': str(dst), 'train': 'train/images', 'val': 'val/images',
        'test': 'test/images', 'nc': 2, 'names': ['module', 'busbar']}))
    print(f'{tag}: {n_img} images linked, {n_lab} label files -> {cfg}')
    if n_img == 0:
        print('WARNING: no images matched. Check that MyDrive/evb/images filenames '
              'match the label stems exactly.')
    return cfg

DATA_ORIG = build_dataset(LABELS_ORIG, 'orig')

## 2. Matched-budget seed reruns

The paper currently compares YOLO11n at 150 epochs / 640 px / SGD against RF-DETR-Nano
at 60 epochs / 384 px, which are the two architecture defaults. That confounds
architecture with training budget. Here both are trained for the **same number of
epochs at the closest resolution each architecture accepts** (RF-DETR requires a
multiple of 56, so 672 is used against YOLO's 640), with **three seeds each**, so the
reported margin can carry a standard deviation.

Reduce `EPOCHS` first if you want a quick smoke test; 100 is the intended setting.

In [ ]:
EPOCHS   = 60   # matched budget; see the GPU table above
SEEDS    = [0, 1, 2]
YOLO_IMG = 640
RFD_IMG  = 672   # RF-DETR requires a multiple of 56; nearest to 640

from ultralytics import YOLO
import json, time

results = {}
for seed in SEEDS:
    name = f'yolo11n_e{EPOCHS}_s{seed}'
    out = ROOT / 'runs' / name
    if (out / 'weights' / 'last.pt').exists():
        print('skip (done):', name); continue
    t0 = time.time()
    m = YOLO('yolo11n.pt')
    m.train(data=str(DATA_ORIG), epochs=EPOCHS, imgsz=YOLO_IMG, seed=seed,
            project=str(ROOT / 'runs'), name=name, exist_ok=True,
            optimizer='SGD', patience=EPOCHS + 1, verbose=False)  # never early-stop
    print(f'{name} done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# RF-DETR needs COCO-format annotations. Polygons MUST be converted to boxes:
# dropping them is the bug that produced a bogus 0.724 in an earlier experiment.
import json, pathlib
from PIL import Image

def yolo_to_coco(ds_dir, split):
    ds = pathlib.Path(ds_dir); img_dir = ds / split / 'images'; lab_dir = ds / split / 'labels'
    out = {'images': [], 'annotations': [],
           'categories': [{'id': 1, 'name': 'module'}, {'id': 2, 'name': 'busbar'}]}
    ann_id = 1
    for i, ip in enumerate(sorted(img_dir.glob('*')), start=1):
        W, H = Image.open(ip).size
        out['images'].append({'id': i, 'file_name': ip.name, 'width': W, 'height': H})
        lp = lab_dir / (ip.stem + '.txt')
        if not lp.exists():
            continue
        for line in lp.read_text().splitlines():
            f = line.split()
            if len(f) < 5:
                continue
            c = int(float(f[0])); v = [float(x) for x in f[1:]]
            if len(v) == 4:
                xc, yc, w, h = v; x0, y0 = xc - w/2, yc - h/2
            elif len(v) >= 6 and len(v) % 2 == 0:      # polygon -> enclosing box
                xs, ys = v[0::2], v[1::2]
                x0, y0, w, h = min(xs), min(ys), max(xs)-min(xs), max(ys)-min(ys)
            else:
                continue
            out['annotations'].append({'id': ann_id, 'image_id': i, 'category_id': c + 1,
                                       'bbox': [x0*W, y0*H, w*W, h*H],
                                       'area': w*W*h*H, 'iscrowd': 0})
            ann_id += 1
    (ds / split / '_annotations.coco.json').write_text(json.dumps(out))
    print(f'{split}: {len(out["images"])} images, {len(out["annotations"])} boxes')

for s in ('train', 'val', 'test'):
    yolo_to_coco('/content/ds_orig', s)

In [ ]:
from rfdetr import RFDETRNano
import shutil

# RF-DETR expects train/valid/test directory names
base = pathlib.Path('/content/rfdetr_orig')
if base.exists(): shutil.rmtree(base)
for src, dst in (('train','train'), ('val','valid'), ('test','test')):
    (base/dst).mkdir(parents=True, exist_ok=True)
    for p in pathlib.Path(f'/content/ds_orig/{src}/images').glob('*'):
        os.symlink(p, base/dst/p.name)
    shutil.copy(f'/content/ds_orig/{src}/_annotations.coco.json', base/dst/'_annotations.coco.json')

for seed in SEEDS:
    name = f'rfdetr_e{EPOCHS}_s{seed}'
    out = ROOT / 'runs' / name
    if (out / 'checkpoint_best_ema.pth').exists():
        print('skip (done):', name); continue
    out.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    RFDETRNano().train(dataset_dir=str(base), epochs=EPOCHS, batch_size=8,
                       grad_accum_steps=2, lr=1e-4, resolution=RFD_IMG,
                       output_dir=str(out))
    print('done:', name)

## 3. Model-based positive control

`scripts/data_prep/synth_convention_shift.py` relabels a consensus source to a
sub-component convention. The training-free screen already flags it (11.1 -> 185.8);
this section completes the control for the **model-based** screen by training on the
manipulated corpus and recomputing per-source accuracy. A pass means the manipulated
source, and only it, falls below the robust fence.

In [ ]:
DATA_SHIFT = build_dataset(LABELS_SHIFT, 'shift')

name = f'yolo11n_convshift_e{EPOCHS}'
if not (ROOT / 'runs' / name / 'weights' / 'last.pt').exists():
    YOLO('yolo11n.pt').train(data=str(DATA_SHIFT), epochs=EPOCHS, imgsz=YOLO_IMG, seed=0,
                             project=str(ROOT / 'runs'), name=name, exist_ok=True,
                             optimizer='SGD', patience=EPOCHS + 1, verbose=False)  # never early-stop
print('trained:', name)

In [ ]:
# EVALUATION SPLIT. The repository's `test/` directory is the 43-image single-source
# split, all of it from the convention-divergent source. The paper's cross-facility
# benchmark is multi-source, and the closest available equivalent here is `val/`,
# which spans every source. Sections 3 and 4 therefore evaluate on `val`.
EVAL_SPLIT = 'val'

# Per-source evaluation + the robust screen of Eq. (1)
import numpy as np, supervision as sv, statistics as st
from collections import defaultdict

MTECH = ('aug_', 'tesla_model', 'chevrolet-bolt', 'nissan-leaf')
PREFIX = {'gqljq': 'roboflow_ev-battery-component-detection-gqljq',
          'edfw3': 'roboflow_ev-battery-components-edfw3',
          'ybmvt': 'ev-battery-sample-ybmvt', 'final_mobilenet': 'final_mobilenet_results',
          'automated': 'automated-disassembly', 'ue_rav4': 'ue_rav4_module',
          'battery_comp': 'battery_comp', 'last_exp4': 'last_exp', 'bmw_i3': 'bmw_i'}

def source_of(n):
    n = n.lower()
    for s, p in PREFIX.items():
        if n.startswith(p): return s
    return 'mtech' if any(n.startswith(p) for p in MTECH) else 'other'

def per_source_map(weights, ds_dir, split=EVAL_SPLIT, conf=0.05, cls=0):
    model = YOLO(weights)
    groups = defaultdict(lambda: ([], []))
    idir = pathlib.Path(ds_dir)/split/'images'; ldir = pathlib.Path(ds_dir)/split/'labels'
    for ip in sorted(idir.glob('*')):
        W, H = Image.open(ip).size
        gt = []
        lp = ldir/(ip.stem+'.txt')
        if lp.exists():
            for line in lp.read_text().splitlines():
                f = line.split()
                if len(f) < 5: continue
                c = int(float(f[0])); v = [float(x) for x in f[1:]]
                if len(v) == 4:
                    xc,yc,w,h = v; x0,y0 = xc-w/2, yc-h/2
                elif len(v) >= 6 and len(v)%2 == 0:
                    xs,ys = v[0::2], v[1::2]
                    x0,y0,w,h = min(xs),min(ys),max(xs)-min(xs),max(ys)-min(ys)
                else: continue
                if c == cls: gt.append([x0*W, y0*H, (x0+w)*W, (y0+h)*H])
        r = model.predict(str(ip), conf=conf, verbose=False)[0]
        d = sv.Detections.from_ultralytics(r); d = d[d.class_id == cls]
        g = groups[source_of(ip.name)]
        g[0].append(sv.Detections(xyxy=np.array(gt, dtype=float).reshape(-1,4),
                                  class_id=np.zeros(len(gt), dtype=int)))
        g[1].append(sv.Detections(xyxy=d.xyxy, class_id=np.zeros(len(d), dtype=int),
                                  confidence=d.confidence))
    out = {}
    for src, (gts, prs) in groups.items():
        if sum(len(x) for x in gts) < 20: continue
        out[src] = float(sv.MeanAveragePrecision.from_detections(prs, gts).map50)
    return out

def screen(scores, k=3):
    a = list(scores.values()); med = st.median(a)
    mad = st.median([abs(x-med) for x in a]); fence = med - k*1.4826*mad
    return fence, [s for s,v in scores.items() if v < fence]

for tag, w, ds in (('original', ROOT/'runs'/f'yolo11n_e{EPOCHS}_s0'/'weights'/'last.pt', '/content/ds_orig'),
                   ('manipulated', ROOT/'runs'/name/'weights'/'last.pt', '/content/ds_shift')):
    sc = per_source_map(str(w), ds)
    f, flag = screen(sc)
    print(f'\n== {tag} corpus ==')
    for s, v in sorted(sc.items(), key=lambda t: -t[1]): print(f'  {s:<18}{v:.3f}')
    print(f'  fence(k=3) {f:.3f}  ->  flagged: {flag}')

## 4. Bootstrap confidence intervals

Every number in the results section is currently a point estimate on 225 images, sliced
further per source. This resamples images with replacement to attach a 95% interval to
each reported mAP.

In [ ]:
def bootstrap_map(weights, ds_dir, split=EVAL_SPLIT, n_boot=1000, conf=0.05, cls=0, seed=0):
    model = YOLO(weights); rng = np.random.default_rng(seed)
    idir = pathlib.Path(ds_dir)/split/'images'; ldir = pathlib.Path(ds_dir)/split/'labels'
    G, P = [], []
    for ip in sorted(idir.glob('*')):
        W, H = Image.open(ip).size; gt = []
        lp = ldir/(ip.stem+'.txt')
        if lp.exists():
            for line in lp.read_text().splitlines():
                f = line.split()
                if len(f) < 5: continue
                c = int(float(f[0])); v = [float(x) for x in f[1:]]
                if len(v) == 4: xc,yc,w,h = v; x0,y0 = xc-w/2, yc-h/2
                elif len(v) >= 6 and len(v)%2 == 0:
                    xs,ys = v[0::2],v[1::2]
                    x0,y0,w,h = min(xs),min(ys),max(xs)-min(xs),max(ys)-min(ys)
                else: continue
                if c == cls: gt.append([x0*W,y0*H,(x0+w)*W,(y0+h)*H])
        r = model.predict(str(ip), conf=conf, verbose=False)[0]
        d = sv.Detections.from_ultralytics(r); d = d[d.class_id == cls]
        G.append(sv.Detections(xyxy=np.array(gt,dtype=float).reshape(-1,4),
                               class_id=np.zeros(len(gt),dtype=int)))
        P.append(sv.Detections(xyxy=d.xyxy, class_id=np.zeros(len(d),dtype=int),
                               confidence=d.confidence))
    n = len(G); vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        vals.append(sv.MeanAveragePrecision.from_detections(
            [P[i] for i in idx], [G[i] for i in idx]).map50)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    point = sv.MeanAveragePrecision.from_detections(P, G).map50
    return float(point), float(lo), float(hi)

w = ROOT/'runs'/f'yolo11n_e{EPOCHS}_s0'/'weights'/'last.pt'
p, lo, hi = bootstrap_map(str(w), '/content/ds_orig', n_boot=1000)
print(f'YOLO11n module mAP@50 = {p:.3f}  95% CI [{lo:.3f}, {hi:.3f}]')

## 5. Export for the paper

In [ ]:
summary = {'epochs': EPOCHS, 'seeds': SEEDS,
            'yolo_imgsz': YOLO_IMG, 'rfdetr_imgsz': RFD_IMG}
# add whatever the sections above printed, then:
(ROOT/'runs'/'summary.json').write_text(json.dumps(summary, indent=2))
print('written ->', ROOT/'runs'/'summary.json')
print('Download it and hand it back so the results tables can be updated.')